# Airline Segmentation Notebook

In [1]:
%%capture
%pip install pandas_gbq

In [2]:
#installing packages
import pandas as pd
import numpy as np
from google.cloud import bigquery
import time
from datetime import timedelta
import json
import re
import itertools

import pandas_gbq
import matplotlib.pyplot as plt

In [3]:
#display settings
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)

In [4]:
# --- CONFIG ---
DIM_COLS = [
#    "country_segment",
    "air_dest_type",
    "ap_segment",
#    "stay_segment",
    "cabin_mapping",
    "refundable_seg",
#    "air_itinerary_type_code",
    "carrier_type",
    "holiday_segment",
]

METRIC_UNITS = "gr_units"
METRIC_TTV = "gr_book_amt"
METRIC_FEE = "gr_contr_fee"

In [5]:
#pull BQ data
sql_query = '''
SELECT * FROM `pcln-pl-busdatasci-prod.commercial_strategy.air_pcln_data_template_2022_2025_v3`;

'''

In [6]:
project_id = 'pcln-pl-busdatasci-prod'

# Initialize a BigQuery client
bq_client = bigquery.Client(project=project_id)

# Execute the query
job = bq_client.query(sql_query)
df = job.to_dataframe()

In [7]:
df = df.copy()

units_pos = df[METRIC_UNITS] > 0
col_prodcls = "OFFER_METHOD_CODE"

df["app_units"] = np.where(
    units_pos & (df["app_book"].astype(str).str.upper() == "APP"),
    df[METRIC_UNITS],
    0.0
)

df["pkg_units"] = np.where(
    units_pos & (df["pkg_offer"].astype(str).str.upper() == "Y"),
    df[METRIC_UNITS],
    0.0
)

df["sopq_units"] = np.where(
    units_pos & (df[col_prodcls].astype(str).str.upper() == "SOPQ"),
    df[METRIC_UNITS],
    0.0
)

is_total = (df[DIM_COLS] == "TOTAL").all(axis=1)

df_total = df[is_total].copy()
df_seg   = df[~is_total].copy()



In [21]:
df_seg = df_seg.copy()
df_seg[DIM_COLS] = df_seg[DIM_COLS].fillna("UNKNOWN")

In [22]:
# -----------------------------------
# ORDER OF DIMENSIONS FOR ROLLUP
# -----------------------------------
DIM_ORDER = DIM_COLS.copy()
BASE_KEYS = ["submit_year"]

GROUPING_LEVELS = [[]] + [
    DIM_ORDER[:i]
    for i in range(1, len(DIM_ORDER) + 1)
]

# -----------------------------------
# BUILD STACKED AGG TABLE
# -----------------------------------
out_dfs = []

for dims in GROUPING_LEVELS:
    group_cols = BASE_KEYS + dims

    tmp = (
        df_seg.groupby(group_cols, dropna=False)
        .agg(
            gr_units=(METRIC_UNITS, "sum"),
            gr_book_amt=(METRIC_TTV, "sum"),
            gr_contr_fee=(METRIC_FEE, "sum"),
            app_units=("app_units", "sum"),
            pkg_units=("pkg_units", "sum"),
            sopq_units=("sopq_units", "sum"),
        )
        .reset_index()
    )

    # fill dimensions not used in this level with TOTAL
    for col in DIM_ORDER:
        if col not in tmp.columns:
            tmp[col] = "TOTAL"

    # reorder dimensions consistently
    tmp = tmp[BASE_KEYS + DIM_ORDER + [
        "gr_units",
        "gr_book_amt",
        "gr_contr_fee",
        "app_units",
        "pkg_units",
        "sopq_units",
    ]]

    # pct metrics
    tmp["pct_units_app"] = np.where(
        tmp["gr_units"] > 0,
        tmp["app_units"] / tmp["gr_units"],
        np.nan
    )

    tmp["pct_units_pkg"] = np.where(
        tmp["gr_units"] > 0,
        tmp["pkg_units"] / tmp["gr_units"],
        np.nan
    )

    tmp["pct_units_sopq"] = np.where(
        tmp["gr_units"] > 0,
        tmp["sopq_units"] / tmp["gr_units"],
        np.nan
    )

    # aggregation label
    tmp["aggregation_level"] = "TOTAL" if len(dims) == 0 else "|".join(dims)

    out_dfs.append(tmp)

seg_agg_df = pd.concat(out_dfs, ignore_index=True, sort=False)

In [23]:
seg_agg_df

,submit_year,air_dest_type,ap_segment,cabin_mapping,refundable_seg,carrier_type,holiday_segment,gr_units,gr_book_amt,gr_contr_fee,app_units,pkg_units,sopq_units,pct_units_app,pct_units_pkg,pct_units_sopq,aggregation_level
0,2022,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"9,400,665.0000","2,935,602,924.1665","65,446,554.3431",0.0000,"1,222,584.0000","736,641.0000",0.0000,0.1301,0.0784,TOTAL
1,2023,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"9,329,544.0000","3,000,353,443.8002","93,557,235.0702","605,175.0000","1,253,867.0000","899,412.0000",0.0649,0.1344,0.0964,TOTAL
2,2024,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"9,472,273.0000","3,037,668,454.6000","103,264,626.9428","1,888,003.0000","1,323,573.0000","1,092,502.0000",0.1993,0.1397,0.1153,TOTAL
3,2025,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"8,713,135.0000","2,731,780,215.1600","107,383,624.7071","2,000,765.0000","1,495,416.0000","1,039,044.0000",0.2296,0.1716,0.1193,TOTAL
4,2022,Domestic,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"7,940,697.0000","2,110,378,114.6100","56,766,902.8700",0.0000,"1,032,002.0000","709,029.0000",0.0000,0.1300,0.0893,air_dest_type
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7669,2025,UNKNOWN,short,ECO,N,FSC,Summer Travel,30.0000,"22,534.1300",571.7550,14.0000,0.0000,0.0000,0.4667,0.0000,0.0000,air_dest_type|ap_segment|cabin_mapping|refunda...
7670,2025,UNKNOWN,short,ECO,N,UNKNOWN,Long Weekend,23.0000,"3,841.0200",66.1800,10.0000,0.0000,0.0000,0.4348,0.0000,0.0000,air_dest_type|ap_segment|cabin_mapping|refunda...
7671,2025,UNKNOWN,short,ECO,N,UNKNOWN,Non-Holiday,50.0000,"8,963.8700",203.3700,8.0000,0.0000,0.0000,0.1600,0.0000,0.0000,air_dest_type|ap_segment|cabin_mapping|refunda...
7672,2025,UNKNOWN,short,ECO,N,UNKNOWN,Spring Break,28.0000,"5,649.5400",59.2300,14.0000,1.0000,0.0000,0.5000,0.0357,0.0000,air_dest_type|ap_segment|cabin_mapping|refunda...


## CAGR

In [24]:
import warnings
warnings.filterwarnings("ignore")

In [25]:
METRICS = ["gr_units", "gr_contr_fee", "gr_book_amt"]
YEARS = [2022, 2023, 2024, 2025]
START_YEAR, END_YEAR = 2022, 2025
N_YEARS = END_YEAR - START_YEAR  # 3 years between endpoints

dfg = seg_agg_df.copy()

KEYS = ["aggregation_level"] + DIM_ORDER

# -------------------------------------------------
# Clean & Filter
# -------------------------------------------------
dfg["submit_year"] = pd.to_numeric(dfg["submit_year"], errors="coerce").astype("Int64")
dfg = dfg[dfg["submit_year"].isin(YEARS)].copy()

for m in METRICS:
    dfg[m] = pd.to_numeric(dfg[m], errors="coerce")

# -------------------------------------------------
# 1) Simple CAGR (2022 -> 2025)
# -------------------------------------------------
wide = dfg.pivot_table(
    index=KEYS,
    columns="submit_year",
    values=METRICS,
    aggfunc="sum"
)

wide.columns = [f"{metric}_{year}" for metric, year in wide.columns]
wide = wide.reset_index()

for m in METRICS:
    start = wide.get(f"{m}_{START_YEAR}")
    end   = wide.get(f"{m}_{END_YEAR}")

    wide[f"{m}_cagr_simple_{START_YEAR}_{END_YEAR}"] = np.where(
        (start > 0) & (end > 0),
        (end / start) ** (1 / N_YEARS) - 1,
        np.nan
    )

# -------------------------------------------------
# 2) Regression CAGR (log-linear)
# -------------------------------------------------
def reg_cagr_loglinear(g: pd.DataFrame, metric: str) -> float:
    x = g["submit_year"].astype(float).to_numpy()
    y = pd.to_numeric(g[metric], errors="coerce").to_numpy()

    mask = np.isfinite(x) & np.isfinite(y) & (y > 0)
    x = x[mask]
    y = y[mask]

    if x.size < 2:
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", np.RankWarning)
        slope = np.polyfit(x, np.log(y), 1)[0]

    return float(np.exp(slope) - 1)

reg_rows = []
for keys, g in dfg.groupby(KEYS, dropna=False):
    row = dict(zip(KEYS, keys if isinstance(keys, tuple) else (keys,)))
    for m in METRICS:
        row[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"] = reg_cagr_loglinear(g, m)
    reg_rows.append(row)

reg_df = pd.DataFrame(reg_rows)

# -------------------------------------------------
# Combine Outputs
# -------------------------------------------------
out = wide.merge(reg_df, on=KEYS, how="left")

# Optional: total growth implied by regression
for m in METRICS:
    ann = out[f"{m}_cagr_reg_annual_{START_YEAR}_{END_YEAR}"]
    out[f"{m}_cagr_reg_total_{START_YEAR}_{END_YEAR}"] = np.where(
        ann.notna(),
        (1 + ann) ** N_YEARS - 1,
        np.nan
    )

In [26]:
out["aggregation_level"].value_counts()

aggregation_level
air_dest_type|ap_segment|cabin_mapping|refundable_seg|carrier_type|holiday_segment    1869
air_dest_type|ap_segment|cabin_mapping|refundable_seg|carrier_type                     393
air_dest_type|ap_segment|cabin_mapping|refundable_seg                                  162
air_dest_type|ap_segment|cabin_mapping                                                 102
air_dest_type|ap_segment                                                                20
air_dest_type                                                                            5
TOTAL                                                                                    1
Name: count, dtype: int64

In [35]:
mask = (
    out["air_dest_type"].astype(str).eq("Domestic") &
    out["ap_segment"].astype(str).eq("short") &
    out["cabin_mapping"].astype(str).eq("ECO") &
    out["refundable_seg"].astype(str).eq("N") &
    out["carrier_type"].astype(str).eq("TOTAL") &
    out["holiday_segment"].astype(str).eq("TOTAL")
)

cagr_cols = [
    "aggregation_level",
    f"gr_units_cagr_simple_{START_YEAR}_{END_YEAR}",
    f"gr_units_cagr_reg_annual_{START_YEAR}_{END_YEAR}",
    f"gr_units_cagr_reg_total_{START_YEAR}_{END_YEAR}",
    f"gr_contr_fee_cagr_simple_{START_YEAR}_{END_YEAR}",
    f"gr_contr_fee_cagr_reg_annual_{START_YEAR}_{END_YEAR}",
    f"gr_contr_fee_cagr_reg_total_{START_YEAR}_{END_YEAR}",
    f"gr_book_amt_cagr_simple_{START_YEAR}_{END_YEAR}",
    f"gr_book_amt_cagr_reg_annual_{START_YEAR}_{END_YEAR}",
    f"gr_book_amt_cagr_reg_total_{START_YEAR}_{END_YEAR}",
]

out.loc[
    mask,
    DIM_COLS + cagr_cols
].sort_values("aggregation_level")

,air_dest_type,ap_segment,cabin_mapping,refundable_seg,carrier_type,holiday_segment,aggregation_level,gr_units_cagr_simple_2022_2025,gr_units_cagr_reg_annual_2022_2025,gr_units_cagr_reg_total_2022_2025,gr_contr_fee_cagr_simple_2022_2025,gr_contr_fee_cagr_reg_annual_2022_2025,gr_contr_fee_cagr_reg_total_2022_2025,gr_book_amt_cagr_simple_2022_2025,gr_book_amt_cagr_reg_annual_2022_2025,gr_book_amt_cagr_reg_total_2022_2025
157,Domestic,short,ECO,N,TOTAL,TOTAL,air_dest_type|ap_segment|cabin_mapping|refunda...,-0.1874,-0.1925,-0.4734,0.0123,-0.0092,-0.0273,-0.1577,-0.1617,-0.4109


In [37]:
mask = (
    seg_agg_df["air_dest_type"].astype(str).eq("Domestic") &
    seg_agg_df["ap_segment"].astype(str).eq("short") &
    seg_agg_df["cabin_mapping"].astype(str).eq("ECO") &
    seg_agg_df["refundable_seg"].astype(str).eq("N") &
    seg_agg_df["carrier_type"].astype(str).eq("FSC") &
    seg_agg_df["holiday_segment"].astype(str).eq("Spring Break")
)

seg_agg_df.loc[
    mask,
    [
        "submit_year",
        "aggregation_level",
        "gr_units",
        "gr_contr_fee",
        "gr_book_amt",
        "pct_units_app",
        "pct_units_pkg",
        "pct_units_sopq",
    ] + DIM_COLS
].sort_values(["aggregation_level", "submit_year"])

,submit_year,aggregation_level,gr_units,gr_contr_fee,gr_book_amt,pct_units_app,pct_units_pkg,pct_units_sopq,air_dest_type,ap_segment,cabin_mapping,refundable_seg,carrier_type,holiday_segment
2488,2022,air_dest_type|ap_segment|cabin_mapping|refunda...,377.0000,177.6900,"192,895.1300",0.0000,0.8992,0.0053,Domestic,short,ECO,N,FSC,Spring Break
3894,2023,air_dest_type|ap_segment|cabin_mapping|refunda...,"213,386.0000","4,376,100.3900","73,836,842.8500",0.0000,0.0892,0.2777,Domestic,short,ECO,N,FSC,Spring Break
5400,2024,air_dest_type|ap_segment|cabin_mapping|refunda...,"176,604.0000","4,165,603.2996","61,820,290.0200",0.2395,0.1032,0.3673,Domestic,short,ECO,N,FSC,Spring Break
6648,2025,air_dest_type|ap_segment|cabin_mapping|refunda...,"119,386.0000","4,140,130.9069","46,765,335.7000",0.2210,0.1475,0.3083,Domestic,short,ECO,N,FSC,Spring Break


In [27]:
# Calc baseline

CAGR_FEE_COL = f"gr_contr_fee_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_FEE_SIMPLE_COL = f"gr_contr_fee_cagr_simple_{START_YEAR}_{END_YEAR}"

CAGR_UNITS_COL = f"gr_units_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_UNITS_SIMPLE_COL = f"gr_units_cagr_simple_{START_YEAR}_{END_YEAR}"

CAGR_TTV_COL = f"gr_book_amt_cagr_reg_annual_{START_YEAR}_{END_YEAR}"
CAGR_TTV_SIMPLE_COL = f"gr_book_amt_cagr_simple_{START_YEAR}_{END_YEAR}"

# =========================================================
# 0) Build baseline from full dataset (before filtering)
# =========================================================

base_2025 = seg_agg_df.copy()
base_2025["submit_year"] = pd.to_numeric(base_2025["submit_year"], errors="coerce")

# 2025 + ALL TOTALS across dimensions
base_2025 = base_2025[
    base_2025["submit_year"] == END_YEAR
].copy()

for c in DIM_ORDER:
    base_2025 = base_2025[
        base_2025[c].astype(str).str.strip().str.upper() == "TOTAL"
    ]

# CAGR baseline from regression output
base_cagr = out.copy()

for c in DIM_COLS:
    base_cagr = base_cagr[
        base_cagr[c].astype(str).str.strip().str.upper() == "TOTAL"
    ]

baseline = {
    CAGR_UNITS_COL: base_cagr[CAGR_UNITS_COL].mean(),
    CAGR_TTV_COL: base_cagr[CAGR_TTV_COL].mean(),
    CAGR_FEE_COL: base_cagr[CAGR_FEE_COL].mean(),
    "pct_units_app": base_2025["pct_units_app"].mean(),
    "pct_units_pkg": base_2025["pct_units_pkg"].mean(),
    "pct_units_sopq": base_2025["pct_units_sopq"].mean(),
}

missing_baselines = [k for k, v in baseline.items() if pd.isna(v)]
if missing_baselines:
    raise ValueError(f"Missing baseline values for: {missing_baselines}")

zero_baselines = [
    k for k, v in baseline.items()
    if isinstance(v, (int, float, np.floating)) and v == 0
]
if zero_baselines:
    raise ValueError(f"Baseline value is 0 for: {zero_baselines}")

In [28]:
baseline

{'gr_units_cagr_reg_annual_2022_2025': -0.021041821765631297,
 'gr_book_amt_cagr_reg_annual_2022_2025': -0.020146093455721048,
 'gr_contr_fee_cagr_reg_annual_2022_2025': 0.17166323337183886,
 'pct_units_app': 0.22962630557198987,
 'pct_units_pkg': 0.1716277780615129,
 'pct_units_sopq': 0.11925030428198347}

In [29]:
# =========================================================
# 1) Filter to END_YEAR and join CAGR
# =========================================================
df_2025 = seg_agg_df.copy()
df_2025["submit_year"] = pd.to_numeric(df_2025["submit_year"], errors="coerce")

df_2025 = df_2025[
    df_2025["submit_year"] == END_YEAR
].copy()

keep_cols = KEYS + [
    "gr_units",
    "gr_contr_fee",
    "gr_book_amt",
    "pct_units_app",
    "pct_units_pkg",
    "pct_units_sopq",
]

df_2025 = df_2025[keep_cols].copy()

# Join CAGR metrics
sc = df_2025.merge(
    out[KEYS + [
        CAGR_UNITS_COL,
        CAGR_UNITS_SIMPLE_COL,
        CAGR_FEE_COL,
        CAGR_FEE_SIMPLE_COL,
        CAGR_TTV_COL,
        CAGR_TTV_SIMPLE_COL,
    ]],
    on=KEYS,
    how="left"
)

# Remove TOTAL rows if needed
# for c in DIM_COLS:
#     sc = sc[sc[c].astype(str).str.strip().str.upper() != "TOTAL"]

sc = sc.dropna(subset=[
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL,
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
    "pct_units_app",
    "pct_units_pkg",
    "pct_units_sopq",
])

sc = sc[sc["gr_units"].fillna(0) > 0].copy()

# Segment label
# sc["segment_label"] = sc[DIM_COLS].astype(str).agg(" | ".join, axis=1)

# Calculate % of Total
TOTAL_FILTER = pd.Series(True, index=sc.index)
for c in DIM_COLS:
    TOTAL_FILTER &= sc[c].astype(str).str.strip().str.upper().eq("TOTAL")

total_row = sc.loc[TOTAL_FILTER].iloc[0]

TOTAL_METRICS = [
    "gr_units",
    "gr_contr_fee",
    "gr_book_amt",
]

totals = sc.loc[TOTAL_FILTER, TOTAL_METRICS].iloc[0]

for col in TOTAL_METRICS:
    sc[f"pct_total_{col}"] = sc[col] / totals[col]

In [30]:
# =========================================================
# 2) Normalization vs baseline
# =========================================================

def ratio_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    if avg == 0 or pd.isna(avg):
        return pd.Series(np.nan, index=s.index)
    idx = s / avg
    if clip_low is not None or clip_high is not None:
        idx = idx.clip(lower=clip_low, upper=clip_high)
    return idx

def robust_diff_vs_avg(s: pd.Series, avg: float, clip_low=None, clip_high=None):
    x = pd.to_numeric(s, errors="coerce")
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad == 0 or np.isnan(mad):
        std = np.nanstd(x)
        scale = std if std not in [0, np.nan] else 1.0
    else:
        scale = mad

    z = (x - avg) / scale
    if clip_low is not None or clip_high is not None:
        z = z.clip(lower=clip_low, upper=clip_high)
    return z

RATIO_CLIP_LOW, RATIO_CLIP_HIGH = 0.25, 4.0
DIFF_CLIP_LOW, DIFF_CLIP_HIGH   = -4.0, 4.0

# -----------------------------------------
# Blend SIMPLE + REGRESSION CAGR (RAW)
# -----------------------------------------

sc["gr_units_cagr_blend"] = np.nanmean(
    sc[[CAGR_UNITS_COL, CAGR_UNITS_SIMPLE_COL]],
    axis=1
)

sc["gr_ttv_cagr_blend"] = np.nanmean(
    sc[[CAGR_TTV_COL, CAGR_TTV_SIMPLE_COL]],
    axis=1
)

sc["gr_fee_cagr_blend"] = np.nanmean(
    sc[[CAGR_FEE_COL, CAGR_FEE_SIMPLE_COL]],
    axis=1
)

# Growth normalization (robust vs baseline)
sc["units_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_units_cagr_blend"], baseline[CAGR_UNITS_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

sc["ttv_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_ttv_cagr_blend"], baseline[CAGR_TTV_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

sc["fee_cagr_norm"] = robust_diff_vs_avg(
    sc["gr_fee_cagr_blend"], baseline[CAGR_FEE_COL],
    DIFF_CLIP_LOW, DIFF_CLIP_HIGH
)

# Mix normalization (ratio vs baseline)
sc["app_norm"] = ratio_vs_avg(
    sc["pct_units_app"], baseline["pct_units_app"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["pkg_norm"] = ratio_vs_avg(
    sc["pct_units_pkg"], baseline["pct_units_pkg"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

sc["sopq_norm"] = ratio_vs_avg(
    sc["pct_units_sopq"], baseline["pct_units_sopq"],
    RATIO_CLIP_LOW, RATIO_CLIP_HIGH
)

cols_to_drop = [
    CAGR_UNITS_COL,
    CAGR_UNITS_SIMPLE_COL,
    CAGR_TTV_COL,
    CAGR_TTV_SIMPLE_COL,
    CAGR_FEE_COL,
    CAGR_FEE_SIMPLE_COL,
]

sc = sc.drop(columns=cols_to_drop)

# =========================================================
# 3) Composite Scores
# =========================================================
w_units, w_ttv = 0.5, 0.5
w_app = w_pkg = w_sopq = 1/3
w_growth, w_mix = 0.6, 0.4

sc["growth_score"] = (
    w_units * sc["units_cagr_norm"] +
    w_ttv   * sc["ttv_cagr_norm"]
)

sc["moat_score"] = (
    w_app  * sc["app_norm"] +
    w_pkg  * sc["pkg_norm"] +
    w_sopq * sc["sopq_norm"]
)

sc["composite_score"] = (
    w_growth * sc["growth_score"] +
    w_mix    * sc["moat_score"]
)

In [31]:
sc

,aggregation_level,air_dest_type,ap_segment,cabin_mapping,refundable_seg,carrier_type,holiday_segment,gr_units,gr_contr_fee,gr_book_amt,pct_units_app,pct_units_pkg,pct_units_sopq,pct_total_gr_units,pct_total_gr_contr_fee,pct_total_gr_book_amt,gr_units_cagr_blend,gr_ttv_cagr_blend,gr_fee_cagr_blend,units_cagr_norm,ttv_cagr_norm,fee_cagr_norm,app_norm,pkg_norm,sopq_norm,growth_score,moat_score,composite_score
0,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"8,713,135.0000","107,383,624.7071","2,731,780,215.1600",0.2296,0.1716,0.1193,1.0000,1.0000,1.0000,-0.0230,-0.0219,0.1756,-0.0043,-0.0039,0.0066,1.0000,1.0000,1.0000,-0.0041,1.0000,0.3976
1,air_dest_type,Domestic,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"7,173,074.0000","96,188,635.1695","1,885,093,879.9700",0.2352,0.1775,0.1429,0.8232,0.8957,0.6901,-0.0310,-0.0336,0.1881,-0.0216,-0.0293,0.0278,1.0243,1.0342,1.1983,-0.0255,1.0856,0.4190
2,air_dest_type,Inbound,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"374,126.0000","2,124,978.0855","190,608,186.4400",0.2183,0.0924,0.0144,0.0429,0.0198,0.0698,-0.0898,-0.0728,-0.0194,-0.1488,-0.1151,-0.3220,0.9506,0.5382,0.2500,-0.1319,0.5796,0.1527
3,air_dest_type,International,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"335,784.0000","1,718,924.0857","110,164,132.3600",0.2317,0.0820,0.0005,0.0385,0.0160,0.0403,0.0666,0.0093,0.1991,0.1896,0.0643,0.0463,1.0089,0.4779,0.2500,0.1269,0.5789,0.3077
4,air_dest_type,Outbound,TOTAL,TOTAL,TOTAL,TOTAL,TOTAL,"829,416.0000","7,344,915.3163","545,346,867.0100",0.1856,0.1931,0.0102,0.0952,0.0684,0.1996,0.0655,0.0412,0.1012,0.1871,0.1341,-0.1188,0.8083,1.1250,0.2500,0.1606,0.7278,0.3875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791,air_dest_type|ap_segment|cabin_mapping|refunda...,UNKNOWN,mid_term,ECO,N,UNKNOWN,Spring Break,13.0000,140.3700,"5,451.2500",0.2308,0.0000,0.0000,0.0000,0.0000,0.0000,-0.2014,-0.3471,0.4204,-0.3900,-0.7151,0.4192,1.0050,0.2500,0.2500,-0.5526,0.5017,-0.1309
1796,air_dest_type|ap_segment|cabin_mapping|refunda...,UNKNOWN,short,BEC,N,FSC,Non-Holiday,10.0000,252.8300,"7,232.5200",0.2000,0.0000,0.0000,0.0000,0.0000,0.0000,0.6673,0.3458,1.0773,1.4885,0.8004,1.5259,0.8710,0.2500,0.2500,1.1444,0.4570,0.8695
1808,air_dest_type|ap_segment|cabin_mapping|refunda...,UNKNOWN,short,ECO,N,FSC,Non-Holiday,33.0000,226.8201,"22,417.7300",0.3333,0.0909,0.0000,0.0000,0.0000,0.0000,0.9584,0.5270,2.3083,2.1178,1.1968,3.6002,1.4516,0.5297,0.2500,1.6573,0.7438,1.2919
1812,air_dest_type|ap_segment|cabin_mapping|refunda...,UNKNOWN,short,ECO,N,UNKNOWN,Non-Holiday,50.0000,203.3700,"8,963.8700",0.1600,0.0000,0.0000,0.0000,0.0000,0.0000,-0.3721,-0.5536,-0.2120,-0.7590,-1.1667,-0.6464,0.6968,0.2500,0.2500,-0.9628,0.3989,-0.4181


In [32]:
sc.to_csv("pcln_air_rollup_segments.csv", index=False)